In [ ]:
"""
Full Exploratory Data Analysis (EDA)
======================================
Dissertation: Predicting Revenue Growth and Cost Reduction from Business AI Adoption
Dataset: Global AI Adoption & Workforce Impact (Kaggle, synthetic, 150,000 rows, 35 columns)

This script covers:
  PART A — Full dataset overview: structure, missing values, dtypes
  PART B — Descriptive statistics for all numeric variables
  PART C — Distributions (histograms) for all numeric variables
  PART D — Correlation matrix for all numeric variables
  PART E — Categorical variable frequency breakdowns
  PART F — Target variable deep-dive: revenue_growth_percent & cost_reduction_percent
            (distribution shape, skewness, kurtosis, normality check,
             outlier detection, suitability for regression)
  PART G — Subgroup breakdowns: industry / company_size / region
            (descriptive stats + visualisations for both targets)

Run: python full_eda.py
Outputs saved to ./eda_outputs/
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import os
import warnings
warnings.filterwarnings("ignore")

# ── Setup ─────────────────────────────────────────────────────────────────
DATA_PATH = "ai_company_adoption.csv"   # <-- adjust to your actual filename
OUT_DIR = "eda_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
NAVY = "#1F4E79"
BLUE = "#2E74B5"
TEAL = "#2E8B7F"
AMBER = "#E8A33D"
RED = "#C0392B"
PALETTE = [NAVY, BLUE, TEAL, AMBER, RED, "#7D3C98", "#909497"]

try:
    df = pd.read_csv("ai_company_adoption.csv")
    print(f"✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find '{DATA_PATH}'.\n"
        f"Update DATA_PATH at the top of this script to point to your CSV file."
    )

# Define your two target variables and key grouping variables
TARGET_1 = "revenue_growth_percent"
TARGET_2 = "cost_reduction_percent"
GROUP_VARS = ["industry", "company_size", "region"]  # adjust names if different in your dataset

# ═══════════════════════════════════════════════════════════════════════════
# PART A — DATASET OVERVIEW
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 80)
print("PART A — DATASET OVERVIEW")
print("=" * 80)

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn dtypes:")
print(df.dtypes.value_counts())

print(f"\nMissing values by column (top 15, if any):")
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary = missing_summary[missing_summary["missing_count"] > 0]
if len(missing_summary) > 0:
    print(missing_summary.head(15).to_string())
else:
    print("No missing values found.")

print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Identify numeric vs categorical columns automatically
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

# Save overview to file
with open(f"{OUT_DIR}/A_dataset_overview.txt", "w") as f:
    f.write(f"Dataset shape: {df.shape}\n\n")
    f.write("Dtypes:\n" + str(df.dtypes) + "\n\n")
    f.write("Missing values:\n" + missing_summary.to_string() + "\n\n")
    f.write(f"Duplicates: {df.duplicated().sum()}\n\n")
    f.write(f"Numeric columns: {numeric_cols}\n\n")
    f.write(f"Categorical columns: {categorical_cols}\n")

# ═══════════════════════════════════════════════════════════════════════════
# PART B — DESCRIPTIVE STATISTICS (ALL NUMERIC VARIABLES)
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PART B — DESCRIPTIVE STATISTICS (NUMERIC VARIABLES)")
print("=" * 80)

desc_stats = df[numeric_cols].describe().T
desc_stats["skewness"] = df[numeric_cols].skew()
desc_stats["kurtosis"] = df[numeric_cols].kurtosis()
desc_stats["missing_pct"] = (df[numeric_cols].isnull().sum() / len(df) * 100).round(2)
desc_stats = desc_stats.round(3)

print(desc_stats.to_string())
desc_stats.to_csv(f"{OUT_DIR}/B_descriptive_statistics.csv")
print(f"\nSaved: {OUT_DIR}/B_descriptive_statistics.csv")

# ═══════════════════════════════════════════════════════════════════════════
# PART C — DISTRIBUTIONS (ALL NUMERIC VARIABLES)
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PART C — DISTRIBUTIONS (HISTOGRAMS FOR ALL NUMERIC VARIABLES)")
print("=" * 80)

n_vars = len(numeric_cols)
n_cols_grid = 4
n_rows_grid = int(np.ceil(n_vars / n_cols_grid))

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(n_cols_grid * 4, n_rows_grid * 3.2))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=40, color=BLUE, alpha=0.75, edgecolor="white", linewidth=0.3)
    axes[i].set_title(col, fontsize=9, fontweight="bold")
    axes[i].tick_params(labelsize=7)
    axes[i].axvline(df[col].mean(), color=RED, linestyle="--", linewidth=1, label="mean")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Distributions — All Numeric Variables", fontsize=14, fontweight="bold", y=1.001)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/C_all_distributions.png", dpi=130, bbox_inches="tight")
print(f"Saved: {OUT_DIR}/C_all_distributions.png")
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# PART D — CORRELATION MATRIX (ALL NUMERIC VARIABLES)
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PART D — CORRELATION MATRIX")
print("=" * 80)

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(max(12, len(numeric_cols) * 0.55), max(10, len(numeric_cols) * 0.5)))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.4, linecolor="white",
    annot_kws={"size": 7}, cbar_kws={"label": "Pearson r", "shrink": 0.7}, ax=ax
)
ax.set_title("Full Correlation Matrix — All Numeric Variables", fontsize=13, fontweight="bold", pad=15)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/D_correlation_matrix.png", dpi=130, bbox_inches="tight")
print(f"Saved: {OUT_DIR}/D_correlation_matrix.png")
plt.close()

corr_matrix.round(3).to_csv(f"{OUT_DIR}/D_correlation_matrix.csv")
print(f"Saved: {OUT_DIR}/D_correlation_matrix.csv")

# Flag high correlation pairs (potential multicollinearity, |r| > 0.7)
print("\nHigh correlation pairs (|r| > 0.7, excluding self-correlation):")
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r_val = corr_matrix.iloc[i, j]
        if abs(r_val) > 0.7:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r_val, 3)))

if high_corr_pairs:
    for pair in high_corr_pairs:
        print(f"  {pair[0]} <-> {pair[1]}: r = {pair[2]}")
else:
    print("  None found.")

# ═══════════════════════════════════════════════════════════════════════════
# PART E — CATEGORICAL VARIABLE FREQUENCY BREAKDOWNS
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PART E — CATEGORICAL VARIABLE FREQUENCIES")
print("=" * 80)

with open(f"{OUT_DIR}/E_categorical_frequencies.txt", "w") as f:
    for col in categorical_cols:
        freq = df[col].value_counts()
        freq_pct = (df[col].value_counts(normalize=True) * 100).round(2)
        summary = pd.DataFrame({"count": freq, "pct": freq_pct})
        print(f"\n--- {col} ({df[col].nunique()} unique values) ---")
        print(summary.head(15).to_string())
        f.write(f"\n--- {col} ({df[col].nunique()} unique values) ---\n")
        f.write(summary.to_string() + "\n")

print(f"\nSaved: {OUT_DIR}/E_categorical_frequencies.txt")

# ═══════════════════════════════════════════════════════════════════════════
# PART F — TARGET VARIABLE DEEP-DIVE
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print(f"PART F — TARGET VARIABLE DEEP-DIVE: {TARGET_1} & {TARGET_2}")
print("=" * 80)

targets = [TARGET_1, TARGET_2]
target_report = []

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for idx, target in enumerate(targets):
    data = df[target].dropna()

    # --- Descriptive stats ---
    mean_, median_, std_ = data.mean(), data.median(), data.std()
    skew_, kurt_ = data.skew(), data.kurtosis()
    min_, max_ = data.min(), data.max()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((data < lower_fence) | (data > upper_fence)).sum()
    pct_outliers = round(n_outliers / len(data) * 100, 2)

    # --- Normality check (Shapiro-Wilk on a sample, since N=150k is too large) ---
    sample = data.sample(min(5000, len(data)), random_state=42)
    shapiro_stat, shapiro_p = stats.shapiro(sample)

    # --- Regression suitability check ---
    suitable = "Likely suitable" if abs(skew_) < 2 and pct_outliers < 5 else "Review recommended"

    print(f"\n--- {target} ---")
    print(f"  N (non-null):        {len(data):,}")
    print(f"  Mean:                {mean_:.3f}")
    print(f"  Median:              {median_:.3f}")
    print(f"  Std Dev:             {std_:.3f}")
    print(f"  Min / Max:           {min_:.3f} / {max_:.3f}")
    print(f"  Skewness:            {skew_:.3f}  ({'roughly symmetric' if abs(skew_) < 0.5 else 'moderate skew' if abs(skew_) < 1 else 'high skew'})")
    print(f"  Kurtosis:            {kurt_:.3f}")
    print(f"  Outliers (IQR rule): {n_outliers:,} ({pct_outliers}%)")
    print(f"  Shapiro-Wilk (n=5000 sample): stat={shapiro_stat:.4f}, p={'<.001' if shapiro_p < 0.001 else round(shapiro_p, 4)}")
    print(f"  Regression suitability: {suitable}")

    target_report.append({
        "target": target, "n": len(data), "mean": round(mean_, 3), "median": round(median_, 3),
        "std": round(std_, 3), "min": round(min_, 3), "max": round(max_, 3),
        "skewness": round(skew_, 3), "kurtosis": round(kurt_, 3),
        "pct_outliers": pct_outliers, "shapiro_p": shapiro_p, "suitability": suitable
    })

    # --- Plots: histogram, boxplot, Q-Q plot ---
    axes[idx, 0].hist(data, bins=50, color=PALETTE[idx], alpha=0.8, edgecolor="white", linewidth=0.3)
    axes[idx, 0].axvline(mean_, color=RED, linestyle="--", linewidth=1.5, label=f"mean={mean_:.1f}")
    axes[idx, 0].axvline(median_, color="black", linestyle=":", linewidth=1.5, label=f"median={median_:.1f}")
    axes[idx, 0].set_title(f"{target} — Distribution", fontsize=10, fontweight="bold")
    axes[idx, 0].legend(fontsize=8)

    axes[idx, 1].boxplot(data, vert=True, patch_artist=True,
                          boxprops=dict(facecolor=PALETTE[idx], alpha=0.7))
    axes[idx, 1].set_title(f"{target} — Boxplot\n({pct_outliers}% outliers via IQR)", fontsize=10, fontweight="bold")

    stats.probplot(data, dist="norm", plot=axes[idx, 2])
    axes[idx, 2].set_title(f"{target} — Q-Q Plot", fontsize=10, fontweight="bold")
    axes[idx, 2].get_lines()[0].set_markerfacecolor(PALETTE[idx])
    axes[idx, 2].get_lines()[0].set_markeredgecolor(PALETTE[idx])
    axes[idx, 2].get_lines()[0].set_markersize(3)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/F_target_deep_dive.png", dpi=140, bbox_inches="tight")
print(f"\nSaved: {OUT_DIR}/F_target_deep_dive.png")
plt.close()

target_report_df = pd.DataFrame(target_report)
target_report_df.to_csv(f"{OUT_DIR}/F_target_variable_report.csv", index=False)
print(f"Saved: {OUT_DIR}/F_target_variable_report.csv")

print("\n--- Suggested dissertation text (EDA chapter) ---")
for row in target_report:
    print(
        f"\n'{row['target']} exhibited a mean of {row['mean']} (SD = {row['std']}), with a skewness of "
        f"{row['skewness']} indicating {'an approximately symmetric' if abs(row['skewness']) < 0.5 else 'a moderately skewed'} "
        f"distribution. Approximately {row['pct_outliers']}% of observations were flagged as outliers using the "
        f"IQR rule. {row['suitability']} for regression-based modelling given the distributional shape observed.'"
    )

# ═══════════════════════════════════════════════════════════════════════════
# PART G — SUBGROUP BREAKDOWNS: industry / company_size / region
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PART G — SUBGROUP BREAKDOWNS (industry / company_size / region)")
print("=" * 80)

available_group_vars = [g for g in GROUP_VARS if g in df.columns]
if len(available_group_vars) < len(GROUP_VARS):
    missing_groups = set(GROUP_VARS) - set(available_group_vars)
    print(f"\n⚠ These grouping variables were not found in the dataset: {missing_groups}")
    print(f"   Available columns: {list(df.columns)}")
    print(f"   Update GROUP_VARS at the top of this script with the correct column names.")

for group_var in available_group_vars:
    print(f"\n{'─' * 70}")
    print(f"BREAKDOWN BY: {group_var}")
    print(f"{'─' * 70}")

    grouped = df.groupby(group_var)[targets].agg(["mean", "median", "std", "count"]).round(2)
    print(grouped.to_string())
    grouped.to_csv(f"{OUT_DIR}/G_breakdown_by_{group_var}.csv")
    print(f"Saved: {OUT_DIR}/G_breakdown_by_{group_var}.csv")

    # Grouped bar chart — mean of both targets per category
    means = df.groupby(group_var)[targets].mean().sort_values(TARGET_1, ascending=False)

    fig, ax = plt.subplots(figsize=(max(8, len(means) * 1.1), 5.5))
    x = np.arange(len(means))
    width = 0.35

    bars1 = ax.bar(x - width/2, means[TARGET_1], width, label=TARGET_1, color=NAVY, alpha=0.85)
    bars2 = ax.bar(x + width/2, means[TARGET_2], width, label=TARGET_2, color=AMBER, alpha=0.85)

    ax.set_xlabel(group_var.replace("_", " ").title(), fontsize=10)
    ax.set_ylabel("Mean predicted value (%)", fontsize=10)
    ax.set_title(f"Mean {TARGET_1} vs {TARGET_2} by {group_var.replace('_', ' ').title()}",
                 fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(means.index, rotation=35, ha="right", fontsize=9)
    ax.legend(fontsize=9)
    ax.axhline(0, color="grey", linewidth=0.8)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/G_chart_{group_var}.png", dpi=130, bbox_inches="tight")
    print(f"Saved: {OUT_DIR}/G_chart_{group_var}.png")
    plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# PART G(ii) — Combined subgroup summary table (for RQ3 write-up)
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PART G(ii) — COMBINED SUBGROUP SUMMARY (for RQ3)")
print("=" * 80)

summary_rows = []
for group_var in available_group_vars:
    means = df.groupby(group_var)[targets].mean()
    top_revenue = means[TARGET_1].idxmax()
    top_cost = means[TARGET_2].idxmax()
    summary_rows.append({
        "grouping_variable": group_var,
        f"highest_{TARGET_1}_category": top_revenue,
        f"highest_{TARGET_1}_value": round(means.loc[top_revenue, TARGET_1], 2),
        f"highest_{TARGET_2}_category": top_cost,
        f"highest_{TARGET_2}_value": round(means.loc[top_cost, TARGET_2], 2),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(f"{OUT_DIR}/G_RQ3_summary_table.csv", index=False)
print(f"\nSaved: {OUT_DIR}/G_RQ3_summary_table.csv")

# ═══════════════════════════════════════════════════════════════════════════
# DONE
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print(f"EDA COMPLETE — all outputs saved to ./{OUT_DIR}/")
print("=" * 80)
print(f"""
Files generated:
  A_dataset_overview.txt          — shape, dtypes, missing values
  B_descriptive_statistics.csv    — full stats table, all numeric vars
  C_all_distributions.png         — histogram grid, all numeric vars
  D_correlation_matrix.png/.csv   — full correlation heatmap + data
  E_categorical_frequencies.txt   — frequency tables, all categorical vars
  F_target_deep_dive.png          — histogram/boxplot/Q-Q for both targets
  F_target_variable_report.csv    — regression suitability report
  G_breakdown_by_*.csv            — subgroup stats (industry/size/region)
  G_chart_*.png                   — grouped bar charts (RQ3 visuals)
  G_RQ3_summary_table.csv         — condensed summary for write-up
""")

In [ ]:
print(df[['num_ai_tools_used','ai_projects_active','ai_adoption_rate']].corr())